# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described using a Croissant schema and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("\nDataset Title:")
print(metadata.name)
print("\nDescription:")
print(metadata.description)


## 2. Data Overview
Review the available record sets, fields, and their `@id`s. This helps identify how the dataset is organized and what data is available for analysis.

In [ ]:
# List all record sets and their @id and fields
print("\nRecord sets with their @id and fields:")

record_sets = list(dataset.record_sets)
record_set_ids = []
for record_set in record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, data type: {field.data_type})")
    print()


### Preview a sample record from each record set
Below we print the first record from each record set, referencing the record set by its `@id`.

In [ ]:
for record_set in record_sets:
    print(f"\nFirst record from record set '{record_set.name}' (@id: {record_set.id}):")
    records = dataset.records(record_set=record_set.id)
    try:
        first_record = next(records)
        print(first_record)
    except StopIteration:
        print("  No records found.")


## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for further analysis. We will refer to each record set by its unique `@id` as required by Croissant and `mlcroissant`.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"Loaded record set: {record_set.name} (@id: {record_set.id}), {len(df)} records.")

# Choose one record set for subsequent demonstration (if multiple, pick the first)
main_record_set_id = record_set_ids[0]

print(f"\nColumns in main record set ({main_record_set_id}):")
print(list(dataframes[main_record_set_id].columns))
print("\nPreview:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some typical data processing steps, such as filtering, normalization, and group-wise aggregation. All references use the corresponding `@id` for both record set and fields.

In [ ]:
# Identify numeric fields in the main DataFrame
df = dataframes[main_record_set_id]
numeric_fields = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Numeric fields detected:", numeric_fields)

# If there are numeric fields, pick the first one for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # This is the @id of the field/column
    threshold = df[numeric_field_id].mean()  # For demonstration use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field, if available
    categorical_fields = df.select_dtypes(include=["object"]).columns.tolist()
    group_field = None
    for col in categorical_fields:
        if col != numeric_field_id:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped statistics by {group_field}:")
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical field for grouping found.")
else:
    print("No numeric fields available for demonstration.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized values (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field and its normalized version if present
if numeric_fields:
    plt.figure(figsize=(14,6))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.subplot(1,2,2)
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=15, kde=True, color='salmon')
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()

    # If we have group_field, visualize group-wise mean
    if group_field:
        plt.figure(figsize=(8,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} per {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook demonstrated:
- How to load a FAIR²-compliant dataset using the Croissant schema and `mlcroissant`.
- How to inspect record sets and fields via their `@id`.
- How to extract and analyze tabular data, filter and normalize values, and perform group-wise aggregations using field and record set `@id`s.
- How to visualize field distributions and group-wise statistics.

You can now explore and process additional fields or record sets as needed using the same approach.